# BigAlpha 2026 First Submission Factor

赛道：AI 因子挖掘 / 第一版单因子。

这个 Notebook 只定义平台会抽取的 `main(data_source, start_datetime, end_datetime)`。因子思想来自本项目 MLK/GRU-GAT 主线的第一层适配：先把每只股票的分钟级时序压缩成稳定的日频表征，再做当日截面排序，最后等权融合为一个单因子。第一版不在提交环境里重训深度模型，优先保证可运行、无外网、无未来数据、覆盖度稳定。


In [ ]:
from __future__ import annotations

import pandas as pd

try:
    import dai
except ImportError:  # 本地静态检查时允许没有 BigQuant 平台包
    dai = None


BAR1M_TABLE = "bigalpha_2026_stock_bar1m"
REQUIRED_COLUMNS = ["date", "instrument", "factor"]


def _resolve_table(data_source: str | None) -> str:
    """Use the table passed by the judge when available; otherwise fall back to the official bar1m table."""
    if isinstance(data_source, str) and data_source.strip():
        return data_source.strip()
    return BAR1M_TABLE


def _date_filter(start_datetime, end_datetime) -> dict:
    return {"date": [str(start_datetime), str(end_datetime)]}


def _neutralize_sign_and_fill(df: pd.DataFrame) -> pd.DataFrame:
    """Last-mile cleanup after DAI aggregation; uses only same-day cross-sectional information."""
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    out["instrument"] = out["instrument"].astype(str)
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce")

    # Platform will winsorize/standardize, but finite values improve validity and coverage.
    out["factor"] = out.groupby("date", observed=True)["factor"].transform(
        lambda s: s.fillna(s.median()).fillna(0.0)
    )
    out = out[REQUIRED_COLUMNS].sort_values(["date", "instrument"]).reset_index(drop=True)
    return out


def main(data_source=None, start_datetime=None, end_datetime=None):
    """Return a daily BigAlpha factor with exactly date, instrument, factor.

    Factor components, all same-day minute-bar based:
    1. intraday reversal: close below VWAP is treated as favorable mean reversion;
    2. order-book pressure: stronger bid-side depth and better bid/ask price balance are favorable;
    3. liquidity efficiency: more traded amount per trade is favorable after cross-sectional ranking;
    4. low noisy range: smaller intraday high-low range is favorable.

    The four components are transformed to daily cross-sectional ranks and equally weighted.
    This is intentionally a robust first submission rather than a trained GRU-GAT artifact.
    """
    if dai is None:
        raise ImportError("This notebook must run inside BigQuant AIStudio where `dai` is available.")
    if start_datetime is None or end_datetime is None:
        raise ValueError("start_datetime and end_datetime are required by the BigAlpha judge.")

    table = _resolve_table(data_source)
    sql = f"""
    WITH daily AS (
        SELECT
            date::DATE::DATETIME AS date,
            instrument::string AS instrument,
            FIRST(pre_close ORDER BY date) AS pre_close,
            FIRST(open ORDER BY date) AS first_open,
            LAST(close ORDER BY date) AS last_price,
            MAX(high) AS day_high,
            MIN(low) AS day_low,
            SUM(amount) AS day_amount,
            SUM(volume) AS day_volume,
            SUM(num_trades) AS day_trades,
            AVG(close) AS avg_price,
            AVG((bid_price1 - ask_price1) / NULLIF((bid_price1 + ask_price1) / 2.0, 0.0)) AS spread_pressure,
            AVG((bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                - ask_volume1 - ask_volume2 - ask_volume3 - ask_volume4 - ask_volume5)
                / NULLIF(bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                    + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5, 0)) AS depth_imbalance,
            LOG(1.0 + SUM(amount)) AS log_amount
        FROM {table}
        GROUP BY date::DATE, instrument
    ),
    features AS (
        SELECT
            date,
            instrument,
            -(last_price / NULLIF(avg_price, 0.0) - 1.0) AS intraday_reversal,
            0.70 * depth_imbalance + 0.30 * spread_pressure AS order_pressure,
            log_amount AS liquidity_scale,
            -((day_high - day_low) / NULLIF(pre_close, 0.0)) AS low_range
        FROM daily
        WHERE last_price IS NOT NULL
          AND avg_price IS NOT NULL
          AND pre_close IS NOT NULL
    ),
    ranked AS (
        SELECT
            date,
            instrument,
            c_rank(intraday_reversal) AS r_reversal,
            c_rank(order_pressure) AS r_pressure,
            c_rank(liquidity_scale) AS r_liquidity,
            c_rank(low_range) AS r_range
        FROM features
    )
    SELECT
        date,
        instrument,
        0.25 * r_reversal
        + 0.25 * r_pressure
        + 0.25 * r_liquidity
        + 0.25 * r_range AS factor
    FROM ranked
    ORDER BY date, instrument
    """
    df = dai.query(sql, filters=_date_filter(start_datetime, end_datetime), compression=True).df()
    return _neutralize_sign_and_fill(df)


## Local Smoke Check

这一个单元只在人工调试时运行。正式评测会直接导入并调用上面的 `main`，不依赖这里的输出。


In [ ]:
# Optional platform smoke test:
# result = main("bigalpha_2026_stock_bar1m", "2023-01-01 00:00:00", "2023-01-10 23:59:59")
# print(result.head())
# print(result.dtypes)
# print(result.groupby("date")["factor"].agg(["count", "mean", "std"]).head())
